In [69]:
!pip install pyfaidx transformers datasets tqdm

import pandas as pd
import requests
from tqdm import tqdm
from pyfaidx import Fasta
from transformers import BertTokenizer, BertForSequenceClassification
import torch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable


In [70]:
pip install ipywidgets

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [80]:
# ========================
#  📌 第二步：读取 VCF 文件（你需要在 Colab 手动上传）
# ========================
#vcf_filename = "ClinVar_Coding_SNV_PB.vcf"  # 你需要替换为自己的 VCF 文件名
vcf_filename = "ClinVar_NonCoding_SNV_PB.vcf"
''' # 解析 VCF 文件
vcf_data = []
with open(vcf_filename, "r") as file:
    for line in file:
        if not line.startswith("#"):  # 跳过注释行
            fields = line.strip().split("\t")
            chrom, pos, ref, alt = fields[0], int(fields[1]), fields[3], fields[4]
            vcf_data.append([chrom, pos, ref, alt]) '''

# 解析 VCF 文件，同时提取 `INFO` 字段
vcf_data = []
with open(vcf_filename, "r") as file:
    for line in file:
        if not line.startswith("#"):  # 跳过注释行
            fields = line.strip().split("\t")

            # 提取关键字段
            chrom, pos, ref, alt, info = fields[0], int(fields[1]), fields[3], fields[4], fields[7]

            # 将数据存入列表
            vcf_data.append([chrom, pos, ref, alt, info])

# 创建 DataFrame
df_vcf = pd.DataFrame(vcf_data, columns=["CHROM", "POS", "REF", "ALT", "INFO"])


#df_vcf = pd.DataFrame(vcf_data, columns=["CHROM", "POS", "REF", "ALT"])

# 生成 True_Label（1=致病, 0=良性）
#df_vcf["True_Label"] = df_vcf["INFO"]

print("✅ 解析 VCF 完成！")




✅ 解析 VCF 完成！


In [81]:
df_vcf

,CHROM,POS,REF,ALT,INFO
0,6,26093008,G,A,0.0
1,2,19989284,T,C,1.0
2,10,97600167,G,T,1.0
3,2,63492941,C,A,1.0
4,1,45014071,G,C,1.0
...,...,...,...,...,...
118439,15,98939361,G,A,1.0
118440,16,2097318,C,T,1.0
118441,11,57811488,T,G,1.0
118442,1,164560014,G,A,1.0


In [82]:
# ========================
#  📌 第三步：下载 GRCh38 参考基因组
# ========================
!wget -c http://hgdownload.cse.ucsc.edu/goldenpath/hg38/bigZips/hg38.fa.gz
!gunzip -k hg38.fa.gz

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


--2025-04-13 04:51:55--  http://hgdownload.cse.ucsc.edu/goldenpath/hg38/bigZips/hg38.fa.gz
128.114.119.163nload.cse.ucsc.edu (hgdownload.cse.ucsc.edu)... 
connected. to hgdownload.cse.ucsc.edu (hgdownload.cse.ucsc.edu)|128.114.119.163|:80... 
416 Requested Range Not Satisfiablee... 

    The file is already fully retrieved; nothing to do.



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


gzip: hg38.fa already exists; do you wish to overwrite (y or n)? ^C


In [83]:
# 加载参考基因组
genome = Fasta("hg38.fa")

# ========================
#  📌 第四步：提取突变上下游 50bp 序列（共 101bp）
# ========================

def get_sequence(row, flank_size=256):
    try:
        chrom = str(row["CHROM"])
        if not chrom.startswith("chr"):
            chrom = "chr" + chrom

        pos = int(row["POS"])
        start = max(0, pos - flank_size - 1)
        end = pos + flank_size

        # 染色体是否在 genome 中
        if chrom not in genome:
            return None

        seq = genome[chrom][start:end].seq.upper()

        # 检查长度
        if len(seq) != (2 * flank_size + 1):
            return None

        return seq
    except Exception as e:
        print(f"[⚠️ get_sequence] Error: {e}")
        return None


def generate_mutant_sequence(row, flank_size=256):
    try:
        seq = list(row["Context_Sequence"])
        mut_pos = flank_size  # 中央是突变位点

        # 检查 REF/ALT 长度是否为单碱基
        if len(row["REF"]) != 1 or len(row["ALT"]) != 1:
            return None

        # 确保 REF 和参考序列匹配
        if seq[mut_pos] != row["REF"]:
            return None

        seq[mut_pos] = row["ALT"]
        return "".join(seq)
    except Exception as e:
        print(f"[⚠️ generate_mutant_sequence] Error: {e}")
        return None


# 设置窗口长度
window = 256

# 获取上下文序列（参考）
from tqdm.notebook import tqdm
tqdm.pandas()
df_vcf["Context_Sequence"] = df_vcf.progress_apply(lambda row: get_sequence(row, flank_size=window), axis=1)

# 删除无效行（可能是 indel 或边界错误）
df_vcf.dropna(subset=["Context_Sequence"], inplace=True)

# 生成突变序列
df_vcf["Mutant_Sequence"] = df_vcf.progress_apply(lambda row: generate_mutant_sequence(row, flank_size=window), axis=1)

# 删除突变失败的行
df_vcf.dropna(subset=["Mutant_Sequence"], inplace=True)

print(f"✅ 成功生成上下文序列和突变序列，共 {len(df_vcf)} 条有效记录")



# ========================
#  📌 第五步：转换为 DNABERT2 的 k-mer 格式
# ========================
from tqdm import tqdm

# 启用 tqdm 进度条
tqdm.pandas()

def generate_kmers(sequence, k=6):
    """ 将序列转换为 k-mer 格式（适用于 DNABERT2） """
    return " ".join([sequence[i:i+k] for i in range(len(sequence) - k + 1)])

# 添加进度条
print("🔄 正在转换 Context_Sequence 为 k-mer 格式...")
df_vcf["Kmer_Sequence"] = df_vcf["Context_Sequence"].progress_apply(lambda x: generate_kmers(x, k=6))

print("🔄 正在转换 Mutant_Sequence 为 k-mer 格式...")
df_vcf["Kmer_Sequence_Mutant"] = df_vcf["Mutant_Sequence"].progress_apply(lambda x: generate_kmers(x, k=6))

# 保存结果
#df_vcf.to_csv("dnabert2_input.csv", index=False)
print("✅ k-mer 转换完成！")




  0%|          | 0/118444 [00:00<?, ?it/s]

  0%|          | 0/118442 [00:00<?, ?it/s]

✅ 成功生成上下文序列和突变序列，共 118441 条有效记录
🔄 正在转换 Context_Sequence 为 k-mer 格式...


100%|██████████| 118441/118441 [00:08<00:00, 13959.54it/s]


🔄 正在转换 Mutant_Sequence 为 k-mer 格式...


100%|██████████| 118441/118441 [00:08<00:00, 14159.94it/s]

✅ k-mer 转换完成！


In [85]:
# ========================
#  📌 使用 DNABERT2 进行原始序列 + 突变序列预测（支持 GPU）
# ========================

import torch
from transformers import BertModel, AutoTokenizer
from tqdm import tqdm
#df_vcf = df_vcf.head(1000)
# ✅ 设置设备（优先使用 GPU）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ 加载模型并移动到 GPU
model = BertModel.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True).to(device)

# ✅ 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

# ✅ 定义预测函数（送入 GPU，结果搬回 CPU）
def predict_sequence(sequence):
    inputs = tokenizer(sequence, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].cpu().tolist()[0]

# ✅ 预测原始序列（Kmer_Sequence）
predictions = []
for _, row in tqdm(df_vcf.iterrows(), total=len(df_vcf), desc="Predicting WT with DNABERT2"):
    pred = predict_sequence(row["Kmer_Sequence"])
    predictions.append(pred)
df_vcf["DNABERT2_Predictions"] = predictions

# ✅ 预测突变序列（Kmer_Sequence_Mutant）
predictions_mut = []
for _, row in tqdm(df_vcf.iterrows(), total=len(df_vcf), desc="Predicting Mutant with DNABERT2"):
    pred_m = predict_sequence(row["Kmer_Sequence_Mutant"])
    predictions_mut.append(pred_m)
df_vcf["DNABERT2_Predictions_Mutant"] = predictions_mut

# ✅ 保存结果
df_vcf.to_pickle("dnabert2_predictions_noncoding.pkl")
print("✅ DNABERT2 原始序列 + 突变序列预测完成，结果已保存为：dnabert2_predictions.pkl")

Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['embeddings.position_embeddings.weight', 'encoder.layer.0.attention.self.key.bias', 'encoder.layer.0.attention.self.key.weight', 'encoder.layer.0.attention.self.query.bias', 'encoder.layer.0.attention.self.query.weight', 'encoder.layer.0.attention.self.value.bias', 'encoder.layer.0.attention.self.value.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.output.LayerNorm.bias', 'encoder.layer.0.output.LayerNorm.weight', 'encoder.layer.0.output.dense.bias', 'encoder.layer.0.output.dense.weight', 'encoder.layer.1.attention.self.key.bias', 'encoder.layer.1.attention.self.key.weight', 'encoder.layer.1.attention.self.query.bias', 'encoder.layer.1.attention.self.query.weight', 'encoder.layer.1.attention.self.value.bias', 'encoder.layer.1.attention.self.value.weight', 'encoder.layer.1.intermediate.dense.b

✅ DNABERT2 原始序列 + 突变序列预测完成，结果已保存为：dnabert2_predictions.pkl


In [32]:
df_vcf111 = df_vcf

In [86]:
import numpy as np
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
#df_vcf = pd.read_pickle("dnabert2_predictions.pkl")
# Step 1️⃣ 计算 delta 向量的欧几里得距离
def compute_distance(row):
    v1 = np.array(row["DNABERT2_Predictions"])
    v2 = np.array(row["DNABERT2_Predictions_Mutant"])
    return np.linalg.norm(v1 - v2)

df_vcf["delta_score"] = df_vcf.apply(compute_distance, axis=1)

# Step 2️⃣ 整理标签
df_vcf["label"] = df_vcf["INFO"].astype(float).astype(int)

# Step 3️⃣ 计算 AUC
auc = roc_auc_score(df_vcf["label"], df_vcf["delta_score"])
print(f"✅ Zero-shot AUC: {auc:.4f}")

✅ Zero-shot AUC: 0.4747


In [15]:
df_vcf = df_vcf111

In [87]:
df_vcf

,CHROM,POS,REF,ALT,INFO,Context_Sequence,Mutant_Sequence,Kmer_Sequence,Kmer_Sequence_Mutant,DNABERT2_Predictions,DNABERT2_Predictions_Mutant,delta_score,label
0,6,26093008,G,A,0.0,GAACTACTACCCCCAGAACATCACCATGAAGTGGCTGAAGGATAAG...,GAACTACTACCCCCAGAACATCACCATGAAGTGGCTGAAGGATAAG...,GAACTA AACTAC ACTACT CTACTA TACTAC ACTACC CTAC...,GAACTA AACTAC ACTACT CTACTA TACTAC ACTACC CTAC...,"[0.039148930460214615, 0.26259365677833557, 0....","[0.04123707860708237, 0.263732373714447, 0.261...",0.082792,0
1,2,19989284,T,C,1.0,AAACACCATAAAATGTTTTGGTAGTTTTCCCTTTAAAATAATCAGA...,AAACACCATAAAATGTTTTGGTAGTTTTCCCTTTAAAATAATCAGA...,AAACAC AACACC ACACCA CACCAT ACCATA CCATAA CATA...,AAACAC AACACC ACACCA CACCAT ACCATA CCATAA CATA...,"[0.05879531800746918, 0.27891358733177185, 0.2...","[0.061121921986341476, 0.27754226326942444, 0....",0.107472,1
2,10,97600167,G,T,1.0,TTGGCTTCTGTGTGGATTCTCTCTGTCGTGCGGGCTCTCTGGGACT...,TTGGCTTCTGTGTGGATTCTCTCTGTCGTGCGGGCTCTCTGGGACT...,TTGGCT TGGCTT GGCTTC GCTTCT CTTCTG TTCTGT TCTG...,TTGGCT TGGCTT GGCTTC GCTTCT CTTCTG TTCTGT TCTG...,"[0.08807440102100372, 0.27099665999412537, 0.2...","[0.08688441663980484, 0.2724774479866028, 0.22...",0.055714,1
3,2,63492941,C,A,1.0,TGTTAGTAAGAGTAATTAAAAGGAATTATTTTCCATTAACCAATTT...,TGTTAGTAAGAGTAATTAAAAGGAATTATTTTCCATTAACCAATTT...,TGTTAG GTTAGT TTAGTA TAGTAA AGTAAG GTAAGA TAAG...,TGTTAG GTTAGT TTAGTA TAGTAA AGTAAG GTAAGA TAAG...,"[0.006512881722301245, 0.27996060252189636, 0....","[0.008575132116675377, 0.2854350805282593, 0.2...",0.083272,1
4,1,45014071,G,C,1.0,TCGGGGCGCGGGGAGATCACTCTGGAAGGTCTGGGGTAGACAAAAG...,TCGGGGCGCGGGGAGATCACTCTGGAAGGTCTGGGGTAGACAAAAG...,TCGGGG CGGGGC GGGGCG GGGCGC GGCGCG GCGCGG CGCG...,TCGGGG CGGGGC GGGGCG GGGCGC GGCGCG GCGCGG CGCG...,"[0.06948746740818024, 0.2592334449291229, 0.25...","[0.068668894469738, 0.25933313369750977, 0.251...",0.086935,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
118439,15,98939361,G,A,1.0,CAGATTGAACAAAGATGATATGCAAACCTCGAAAGAAATTGGCATG...,CAGATTGAACAAAGATGATATGCAAACCTCGAAAGAAATTGGCATG...,CAGATT AGATTG GATTGA ATTGAA TTGAAC TGAACA GAAC...,CAGATT AGATTG GATTGA ATTGAA TTGAAC TGAACA GAAC...,"[0.0220880638808012, 0.2708742022514343, 0.247...","[0.019006121903657913, 0.2704446017742157, 0.2...",0.081419,1
118440,16,2097318,C,T,1.0,TACCCCAGGCGGGAACCACGGCTGCCTGGCCTGAGTCCCGGCCCCT...,TACCCCAGGCGGGAACCACGGCTGCCTGGCCTGAGTCCCGGCCCCT...,TACCCC ACCCCA CCCCAG CCCAGG CCAGGC CAGGCG AGGC...,TACCCC ACCCCA CCCCAG CCCAGG CCAGGC CAGGCG AGGC...,"[0.11102991551160812, 0.22054727375507355, 0.2...","[0.10650981962680817, 0.2219357043504715, 0.22...",0.062743,1
118441,11,57811488,T,G,1.0,ATAATATCCTAGTTTGTCTCACTGTCTAGGGTATTTATTCAACTCT...,ATAATATCCTAGTTTGTCTCACTGTCTAGGGTATTTATTCAACTCT...,ATAATA TAATAT AATATC ATATCC TATCCT ATCCTA TCCT...,ATAATA TAATAT AATATC ATATCC TATCCT ATCCTA TCCT...,"[0.021534116938710213, 0.28580284118652344, 0....","[0.022342974320054054, 0.28371670842170715, 0....",0.070965,1
118442,1,164560014,G,A,1.0,TGCTTCCCAGGAGCCGAGCCGAGGAGCAGAAGAGGAAGAGCCGGGG...,TGCTTCCCAGGAGCCGAGCCGAGGAGCAGAAGAGGAAGAGCCGGGG...,TGCTTC GCTTCC CTTCCC TTCCCA TCCCAG CCCAGG CCAG...,TGCTTC GCTTCC CTTCCC TTCCCA TCCCAG CCCAGG CCAG...,"[0.03975480794906616, 0.28963175415992737, 0.2...","[0.03987179324030876, 0.29011309146881104, 0.2...",0.073043,1


In [90]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

# ✅ 加载已经包含 embeddings 的数据
#df_vcf = pd.read_pickle("dnabert2_predictions.pkl")

# ✅ 构造向量差作为输入特征（mutant - wt）
X = df_vcf.apply(lambda row: np.array(row["DNABERT2_Predictions_Mutant"]) - np.array(row["DNABERT2_Predictions"]), axis=1)
X = np.stack(X.values)

# ✅ 准备标签
y = df_vcf["INFO"].astype(float).astype(int).values

# ✅ 拆分训练和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ✅ 训练 XGBoost 分类器
clf = XGBClassifier(
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1  # 并行加速
)
clf.fit(X_train, y_train)

# ✅ 预测 + AUC
y_prob = clf.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
print(f"🎯 XGBoost AUC: {auc:.4f}")

# ✅ 可选：打印分类报告
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, digits=4))


/home/ubuntu/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [05:40:34] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


🎯 XGBoost AUC: 0.8768
              precision    recall  f1-score   support

           0     0.9291    0.9840    0.9558     21245
           1     0.7140    0.3474    0.4674      2444

    accuracy                         0.9183     23689
   macro avg     0.8216    0.6657    0.7116     23689
weighted avg     0.9069    0.9183    0.9054     23689



In [58]:
pip install xgboost


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.9/253.9 MB 11.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.7/291.7 MB 9.3 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [93]:
df_embed = pd.read_pickle("dnabert2_predictions_noncoding.pkl")
annotation_df = pd.read_csv("ClinVar_NonCoding_SNV_PB.csv", sep="\t")

In [97]:
# 确保 key 一致格式
df_embed["POS"] = df_embed["POS"].astype(str)
annotation_df["POS"] = annotation_df["POS"].astype(str)

# 合并两个表（inner 保证对齐）
df_merged = pd.merge(
    df_embed,
    annotation_df,
    left_on=["CHROM", "POS", "REF", "ALT"],
    right_on=["#CHROM", "POS", "REF", "ALT"],
    how="inner"
)


In [98]:
df_merged

,CHROM_x,POS,REF,ALT,INFO_x,Context_Sequence,Mutant_Sequence,Kmer_Sequence,Kmer_Sequence_Mutant,DNABERT2_Predictions,...,exon,CDS,start_codon,stop_codon,five_prime_UTR,three_prime_UTR,intron,promoter,other,CHROM_y
0,6,26093008,G,A,0.0,GAACTACTACCCCCAGAACATCACCATGAAGTGGCTGAAGGATAAG...,GAACTACTACCCCCAGAACATCACCATGAAGTGGCTGAAGGATAAG...,GAACTA AACTAC ACTACT CTACTA TACTAC ACTACC CTAC...,GAACTA AACTAC ACTACT CTACTA TACTAC ACTACC CTAC...,"[0.039148930460214615, 0.26259365677833557, 0....",...,0,0,0,0,0,0,1,0,0,6
1,2,19989284,T,C,1.0,AAACACCATAAAATGTTTTGGTAGTTTTCCCTTTAAAATAATCAGA...,AAACACCATAAAATGTTTTGGTAGTTTTCCCTTTAAAATAATCAGA...,AAACAC AACACC ACACCA CACCAT ACCATA CCATAA CATA...,AAACAC AACACC ACACCA CACCAT ACCATA CCATAA CATA...,"[0.05879531800746918, 0.27891358733177185, 0.2...",...,0,0,0,0,0,0,1,0,0,2
2,10,97600167,G,T,1.0,TTGGCTTCTGTGTGGATTCTCTCTGTCGTGCGGGCTCTCTGGGACT...,TTGGCTTCTGTGTGGATTCTCTCTGTCGTGCGGGCTCTCTGGGACT...,TTGGCT TGGCTT GGCTTC GCTTCT CTTCTG TTCTGT TCTG...,TTGGCT TGGCTT GGCTTC GCTTCT CTTCTG TTCTGT TCTG...,"[0.08807440102100372, 0.27099665999412537, 0.2...",...,0,0,0,0,0,0,1,0,0,10
3,2,63492941,C,A,1.0,TGTTAGTAAGAGTAATTAAAAGGAATTATTTTCCATTAACCAATTT...,TGTTAGTAAGAGTAATTAAAAGGAATTATTTTCCATTAACCAATTT...,TGTTAG GTTAGT TTAGTA TAGTAA AGTAAG GTAAGA TAAG...,TGTTAG GTTAGT TTAGTA TAGTAA AGTAAG GTAAGA TAAG...,"[0.006512881722301245, 0.27996060252189636, 0....",...,0,0,0,0,0,0,1,0,0,2
4,1,45014071,G,C,1.0,TCGGGGCGCGGGGAGATCACTCTGGAAGGTCTGGGGTAGACAAAAG...,TCGGGGCGCGGGGAGATCACTCTGGAAGGTCTGGGGTAGACAAAAG...,TCGGGG CGGGGC GGGGCG GGGCGC GGCGCG GCGCGG CGCG...,TCGGGG CGGGGC GGGGCG GGGCGC GGCGCG GCGCGG CGCG...,"[0.06948746740818024, 0.2592334449291229, 0.25...",...,0,0,0,0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118436,15,98939361,G,A,1.0,CAGATTGAACAAAGATGATATGCAAACCTCGAAAGAAATTGGCATG...,CAGATTGAACAAAGATGATATGCAAACCTCGAAAGAAATTGGCATG...,CAGATT AGATTG GATTGA ATTGAA TTGAAC TGAACA GAAC...,CAGATT AGATTG GATTGA ATTGAA TTGAAC TGAACA GAAC...,"[0.0220880638808012, 0.2708742022514343, 0.247...",...,0,0,0,0,0,0,1,0,0,15
118437,16,2097318,C,T,1.0,TACCCCAGGCGGGAACCACGGCTGCCTGGCCTGAGTCCCGGCCCCT...,TACCCCAGGCGGGAACCACGGCTGCCTGGCCTGAGTCCCGGCCCCT...,TACCCC ACCCCA CCCCAG CCCAGG CCAGGC CAGGCG AGGC...,TACCCC ACCCCA CCCCAG CCCAGG CCAGGC CAGGCG AGGC...,"[0.11102991551160812, 0.22054727375507355, 0.2...",...,0,0,0,0,0,0,1,0,0,16
118438,11,57811488,T,G,1.0,ATAATATCCTAGTTTGTCTCACTGTCTAGGGTATTTATTCAACTCT...,ATAATATCCTAGTTTGTCTCACTGTCTAGGGTATTTATTCAACTCT...,ATAATA TAATAT AATATC ATATCC TATCCT ATCCTA TCCT...,ATAATA TAATAT AATATC ATATCC TATCCT ATCCTA TCCT...,"[0.021534116938710213, 0.28580284118652344, 0....",...,0,0,0,0,0,0,1,0,0,11
118439,1,164560014,G,A,1.0,TGCTTCCCAGGAGCCGAGCCGAGGAGCAGAAGAGGAAGAGCCGGGG...,TGCTTCCCAGGAGCCGAGCCGAGGAGCAGAAGAGGAAGAGCCGGGG...,TGCTTC GCTTCC CTTCCC TTCCCA TCCCAG CCCAGG CCAG...,TGCTTC GCTTCC CTTCCC TTCCCA TCCCAG CCCAGG CCAG...,"[0.03975480794906616, 0.28963175415992737, 0.2...",...,0,0,0,0,0,0,1,0,0,1


In [101]:
# 你的目标列
category_columns = [
    "exon", "CDS", "start_codon", "stop_codon",
    "five_prime_UTR", "three_prime_UTR", "intron", "promoter", "other"
]

# 统计每列中 1（属于该类别）和 0（不属于）的数量
summary = []
for col in category_columns:
    count_1 = (df_merged[col] == 1).sum()
    count_0 = (df_merged[col] == 0).sum()
    summary.append({"Region": col, "Positive (1)": count_1, "Negative (0)": count_0})

# 转为表格查看
df_region_counts = pd.DataFrame(summary)
print(df_region_counts)


            Region  Positive (1)  Negative (0)
0             exon         14972        103469
1              CDS             0        118441
2      start_codon             0        118441
3       stop_codon             0        118441
4   five_prime_UTR          2518        115923
5  three_prime_UTR         12109        106332
6           intron         98897         19544
7         promoter             0        118441
8            other          3027        115414


In [107]:
# Step 5️⃣ 分组保存子集为 .pkl
for col in category_columns:
    subset = df_merged[df_merged[col] == 1].copy()
    subset.to_pickle(f"df_{col}_subset.pkl")
    print(f"✅ Saved df_{col}_subset.pkl with {len(subset)} records")

✅ Saved df_exon_subset.pkl with 14972 records
✅ Saved df_CDS_subset.pkl with 0 records
✅ Saved df_start_codon_subset.pkl with 0 records
✅ Saved df_stop_codon_subset.pkl with 0 records
✅ Saved df_five_prime_UTR_subset.pkl with 2518 records
✅ Saved df_three_prime_UTR_subset.pkl with 12109 records
✅ Saved df_intron_subset.pkl with 98897 records
✅ Saved df_promoter_subset.pkl with 0 records
✅ Saved df_other_subset.pkl with 3027 records


In [109]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

# ✅ 加载已经包含 embeddings 的数据
df_vcf = pd.read_pickle("df_intron_subset.pkl")

# ✅ 构造向量差作为输入特征（mutant - wt）
X = df_vcf.apply(lambda row: np.array(row["DNABERT2_Predictions_Mutant"]) - np.array(row["DNABERT2_Predictions"]), axis=1)
X = np.stack(X.values)

# ✅ 准备标签
y = df_vcf["INFO_x"].astype(float).astype(int).values

# ✅ 拆分训练和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ✅ 训练 XGBoost 分类器
clf = XGBClassifier(
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1  # 并行加速
)
clf.fit(X_train, y_train)

# ✅ 预测 + AUC
y_prob = clf.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
print(f"🎯 XGBoost AUC: {auc:.4f}")

# ✅ 可选：打印分类报告
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, digits=4))


/home/ubuntu/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [06:10:22] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


🎯 XGBoost AUC: 0.8827
              precision    recall  f1-score   support

           0     0.9213    0.9805    0.9500     17409
           1     0.7287    0.3851    0.5039      2371

    accuracy                         0.9091     19780
   macro avg     0.8250    0.6828    0.7269     19780
weighted avg     0.8982    0.9091    0.8965     19780



In [110]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

# ✅ 要处理的子集文件
file_names = [
    "df_exon_subset.pkl",
    "df_five_prime_UTR_subset.pkl",
    "df_three_prime_UTR_subset.pkl",
    "df_intron_subset.pkl",
    "df_other_subset.pkl"
]

for file in file_names:
    print(f"\n==================\n📂 Processing: {file}\n==================")

    try:
        # 加载数据
        df = pd.read_pickle(file)

        # 特征构造
        X = df.apply(lambda row: np.array(row["DNABERT2_Predictions_Mutant"]) - np.array(row["DNABERT2_Predictions"]), axis=1)
        X = np.stack(X.values)

        # 标签构造
        y = df["INFO_x"].astype(float).astype(int).values

        # 跳过只包含单类的情况
        if len(np.unique(y)) < 2:
            print(f"❌ Skipped: Only one class in labels.")
            continue

        # 划分训练 / 测试集
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42
        )

        # 训练 XGBoost
        clf = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42, n_jobs=-1)
        clf.fit(X_train, y_train)

        # 评估 AUC
        y_prob = clf.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        print(f"🎯 AUC: {auc:.4f}")

        # 分类报告
        y_pred = clf.predict(X_test)
        print(classification_report(y_test, y_pred, digits=4))

    except Exception as e:
        print(f"⚠️ Error processing {file}: {e}")



📂 Processing: df_exon_subset.pkl


/home/ubuntu/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [06:12:28] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


🎯 AUC: 0.5416
              precision    recall  f1-score   support

           0     0.9883    1.0000    0.9941      2960
           1     0.0000    0.0000    0.0000        35

    accuracy                         0.9883      2995
   macro avg     0.4942    0.5000    0.4971      2995
weighted avg     0.9768    0.9883    0.9825      2995


📂 Processing: df_five_prime_UTR_subset.pkl


/usr/lib/python3/dist-packages/sklearn/metrics/_classification.py:1221: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/ubuntu/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [06:12:33] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


🎯 AUC: 0.7769
              precision    recall  f1-score   support

           0     0.9742    1.0000    0.9869       491
           1     0.0000    0.0000    0.0000        13

    accuracy                         0.9742       504
   macro avg     0.4871    0.5000    0.4935       504
weighted avg     0.9491    0.9742    0.9615       504


📂 Processing: df_three_prime_UTR_subset.pkl


...

/usr/lib/python3/dist-packages/sklearn/metrics/_classification.py:1221: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/ubuntu/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [06:13:02] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


🎯 AUC: 0.8827
              precision    recall  f1-score   support

           0     0.9213    0.9805    0.9500     17409
           1     0.7287    0.3851    0.5039      2371

    accuracy                         0.9091     19780
   macro avg     0.8250    0.6828    0.7269     19780
weighted avg     0.8982    0.9091    0.8965     19780


📂 Processing: df_other_subset.pkl


/home/ubuntu/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [06:13:11] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


🎯 AUC: 0.6322
              precision    recall  f1-score   support

           0     0.9472    1.0000    0.9729       574
           1     0.0000    0.0000    0.0000        32

    accuracy                         0.9472       606
   macro avg     0.4736    0.5000    0.4864       606
weighted avg     0.8972    0.9472    0.9215       606



/usr/lib/python3/dist-packages/sklearn/metrics/_classification.py:1221: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
